In [14]:
import os
import torchvision
import torch
print(torch.version)
print(torch.cuda.is_available())
print(torch.version.cuda)
import pandas as pd
from dataclasses import dataclass, field
from pathlib import Path
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import shutil
import glob


<module 'torch.version' from '/home/evergreen/whisper-transcription-env/lib/python3.12/site-packages/torch/version.py'>
True
12.8


In [8]:
@dataclass
class TrainingConfig:
    input_path: str = "data/training_pairs_df.csv"
    output_dir: str = "./stitcher_model"
    
    model_name: str = "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length: int = 512
    
    
    lora_rank: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    target_modules: list = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])
    
    num_epochs: int = 3
    batch_size: int = 16
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    warmup_steps: int = 50
    weight_decay: float = 0.01
    lr_scheduler: str = "cosine"

    sample_n: int = 300
    
    logging_steps: int = 10
    save_steps: int = 100

In [9]:

def load_dataframe(path: str) -> pd.DataFrame:

    path = Path(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    elif path.suffix == ".parquet":
        return pd.read_parquet(path)

SYSTEM_PROMPT = (
    "You are a transcript stitcher. You receive two overlapping transcript "
    "fragments. Output ONLY the merged text. No commentary, no explanation, "
    "no preamble. Raw merged text only."
)

def format_chat_example(row: pd.Series) -> dict:

    user_content = (
        f"[FRAGMENT 1]\n{row['frag_1']}\n\n"
        f"[FRAGMENT 2]\n{row['frag_2']}"
    )
    
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": row["test_string"]},
        ]
    }


def prepare_dataset(config: TrainingConfig):
    
    df = load_dataframe(config.input_path)
    df = df.sample(n=config.sample_n, random_state = 1).reset_index(drop = True)
    formatted = [format_chat_example(row) for _, row in df.iterrows()]
    dataset = Dataset.from_list(formatted)
    
    return dataset

In [10]:
def load_model_and_tokenizer(config: TrainingConfig):

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.model_name,
        max_seq_length=config.max_seq_length,
        load_in_4bit=True
    )
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=config.target_modules,
        bias="none",
        use_gradient_checkpointing="unsloth"
    )
    
    return model, tokenizer


In [11]:
def train(config: TrainingConfig):

    dataset = prepare_dataset(config)
    model, tokenizer = load_model_and_tokenizer(config)
    
    def formatting_func(examples):
        convos = examples["conversations"]
        
        if isinstance(convos[0], dict):
            convos = [convos]
    
        texts = []
        
        for convo in convos:
            text = tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False,
            )
            texts.append(text)
        return texts
    
    training_args = SFTConfig(
        output_dir=config.output_dir,
        num_train_epochs=config.num_epochs,
        per_device_train_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        warmup_steps=config.warmup_steps,
        weight_decay=config.weight_decay,
        lr_scheduler_type=config.lr_scheduler,
        logging_steps=config.logging_steps,
        save_steps=config.save_steps,
        save_total_limit=3,
        bf16=True, 
        max_seq_length=config.max_seq_length,
        dataset_num_proc=2,
        packing=False
    )
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        formatting_func=formatting_func,
        args=training_args,
    )
    
    trainer.train()

    model.save_pretrained(config.output_dir)
    tokenizer.save_pretrained(config.output_dir)
    
    return model, tokenizer



In [16]:
config = TrainingConfig()
model, tokenizer = train(config)

model.save_pretrained_gguf(
    "testdir/stitcher_gguf_direct",
    tokenizer,
    quantization_method="q8_0",
)

target_dir = "testdir/stitcher_gguf_direct"

for gguf_file in glob.glob("*.gguf"):
    shutil.move(gguf_file, os.path.join(target_dir, gguf_file))

if os.path.exists("Modelfile"):
    shutil.move("Modelfile", os.path.join(target_dir, "Modelfile"))

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 3080 Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 3 | Total steps = 15
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 4 x 1) = 64
 "-____-"     Trainable parameters = 5,636,096 of 1,241,450,496 (0.45% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,4.352900


...

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:07<00:00, 67.83s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:09<00:00,  9.10s/it]


Unsloth: Merge process complete. Saved to `/home/evergreen/whisper-transcription-env/dev_wsl/whisper_stream_proj/testdir/stitcher_gguf_direct`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['llama-3.2-1b-instruct.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['llama-3.2-1b-instruct.Q8_0.gguf']
Un

In [10]:
from llama_cpp import Llama

llm = Llama(
    model_path="./llama-3.2-1b-instruct.Q8_0.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
    verbose=True
)

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 3080 Laptop GPU, compute capability 8.6, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce RTX 3080 Laptop GPU) - 5748 MiB free
llama_model_loader: loaded meta data with 32 key-value pairs and 147 tensors from ./llama-3.2-1b-instruct.Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Stitcher_Gguf_Direct
llama_model_loader: - kv   3:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   4:           

In [35]:
response = llm.create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": (
                "You are a transcript stitcher. You receive two overlapping transcript "
                "fragments. Output ONLY the merged text. No commentary, no explanation, "
                "no preamble. Raw merged text only."
            )
        },
        {
            "role": "user",
            "content": (
                "[FRAGMENT 1]\n"
                "already exist in the class, the behavior depends. \n\n"
                "[FRAGMENT 2]\n"
                "the behavior depends on the parameter"
            )
        }
    ],
    max_tokens=100,
    temperature=0.1
)

print(repr(response["choices"][0]["message"]["content"]))

Llama.generate: 97 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     395.94 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =     172.13 ms /    12 runs   (   14.34 ms per token,    69.72 tokens per second)
llama_perf_context_print:       total time =     181.36 ms /    13 tokens
llama_perf_context_print:    graphs reused =         12


'already exist in the class the behavior depends on the parameter'


In [14]:
response = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
    max_tokens=50,
    temperature=0.1
)
print(response["choices"][0]["message"]["content"])

Llama.generate: 25 prefix-match hit, remaining 17 prompt tokens to eval
llama_perf_context_print:        load time =     395.94 ms
llama_perf_context_print: prompt eval time =      57.48 ms /    17 tokens (    3.38 ms per token,   295.77 tokens per second)
llama_perf_context_print:        eval time =     117.90 ms /     3 runs   (   39.30 ms per token,    25.45 tokens per second)
llama_perf_context_print:       total time =     179.23 ms /    20 tokens
llama_perf_context_print:    graphs reused =          2


france paris
